In [10]:
import numpy as np
import pandas as pd
import joblib as jb

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans


In [11]:
df = pd.read_csv("bank_transactions.csv")

In [12]:
df.head()

,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR)
0,T1,C5841053,10/1/94,F,JAMSHEDPUR,17819.05,2/8/16,143207,25.0
1,T2,C2142763,4/4/57,M,JHAJJAR,2270.69,2/8/16,141858,27999.0
2,T3,C4417068,26/11/96,F,MUMBAI,17874.44,2/8/16,142712,459.0
3,T4,C5342380,14/9/73,F,MUMBAI,866503.21,2/8/16,142714,2060.0
4,T5,C9031234,24/3/88,F,NAVI MUMBAI,6714.43,2/8/16,181156,1762.5


In [13]:
df.dtypes

TransactionID               object
CustomerID                  object
CustomerDOB                 object
CustGender                  object
CustLocation                object
CustAccountBalance         float64
TransactionDate             object
TransactionTime              int64
TransactionAmount (INR)    float64
dtype: object

In [14]:
df.isnull().sum()

TransactionID                 0
CustomerID                    0
CustomerDOB                3397
CustGender                 1100
CustLocation                151
CustAccountBalance         2369
TransactionDate               0
TransactionTime               0
TransactionAmount (INR)       0
dtype: int64

In [31]:
df['CustomerDOB'].fillna(df['CustomerDOB'].mode()[0],inplace=True)

In [32]:
df['CustGender'].fillna(df['CustGender'].mode()[0],inplace=True)

In [33]:
df['CustLocation'].fillna(df['CustLocation'].mode()[0],inplace=True)

In [34]:
df['CustAccountBalance'].fillna(df['CustAccountBalance'].mean())

0           17819.05
1            2270.69
2           17874.44
3          866503.21
4            6714.43
             ...    
1048562      7635.19
1048563     27311.42
1048564    221757.06
1048565     10117.87
1048566     75734.42
Name: CustAccountBalance, Length: 1048567, dtype: float64

In [19]:
df.isnull().sum()

TransactionID                 0
CustomerID                    0
CustomerDOB                   0
CustGender                    0
CustLocation                  0
CustAccountBalance         2369
TransactionDate               0
TransactionTime               0
TransactionAmount (INR)       0
dtype: int64

In [20]:
df.dtypes

TransactionID               object
CustomerID                  object
CustomerDOB                 object
CustGender                  object
CustLocation                object
CustAccountBalance         float64
TransactionDate             object
TransactionTime              int64
TransactionAmount (INR)    float64
dtype: object

In [21]:
x = df.drop(['TransactionID','CustomerID','CustomerDOB','TransactionDate','TransactionTime'],axis=1)

In [22]:
numerical_cols = x.select_dtypes(include=['int64','float64']).columns.tolist()

In [23]:
categorical_cols = x.select_dtypes(include=['object']).columns.tolist()

In [24]:
numerical_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])

In [25]:
categorical_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])

In [26]:
preprocessor = ColumnTransformer(transformers=[
    ('num',numerical_transformer,numerical_cols),
    ('cat',categorical_transformer,categorical_cols)
])

In [27]:
model = Pipeline(steps=[
    ('pre',preprocessor),('reg',KMeans(n_clusters=3,random_state=42))
])


In [35]:
model.fit(x)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['CustAccountBalance',
                                                   'TransactionAmount (INR)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['CustGender',
                                                   'CustLocation'])])),
                ('reg', KMeans(n_clusters=3, random_state=42))])

In [29]:
cluster = model.predict(x)
print(f'{cluster}')

[0 1 0 ... 1 1 1]


In [30]:
jb.dump(model,'KMeans.pkl')

['KMeans.pkl']

In [49]:
import streamlit as st
import pandas as pd
import joblib as jb

load = jb.load('KMeans.pkl')

st.title("Customer Segmentation prediction")

CustGender = st.selectbox("Gender",["Male","Female","Other"])
CustLocation = st.text_input("CustLocation")
CustAccountBalance = st.number_input("CustAccountBalance")
transaction_amount = st.number_input('TransactionAmount (INR)')

if st.button("Predict"):

    data = {
        "CustGender":[CustGender],
        "CustLocation":[CustLocation],
        "CustAccountBalance":[CustAccountBalance],
        "TransactionAmount (INR)":[transaction_amount]
    }

    df = pd.DataFrame(data)

    prediction = load.predict(df)
    
    if prediction[0] == 0:
        st.success("High Spender 💰")
    else :
     st.success("Low Spender 😐")

2026-04-05 00:12:21.274 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.276 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:12:21.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar